# TP1 (modelos de regresión) — resolución de Ramiro

Notebook generado a partir de `TP1 ramiro.py`.

El objetivo del TP es predecir el peso del bebé al nacer (`weight`, en libras) mediante regresión lineal, evaluando los modelos con las métricas MAE y R².

Los datos (`X_train.csv`, `y_train.csv`, `X_test.csv`) se encuentran en la misma carpeta que este notebook, por lo que se cargan por ruta relativa.

# Carga de los datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Armado de los DataFrames:
predictores = pd.read_csv("X_train.csv")
var_ind = pd.read_csv("y_train.csv")["weight"]
X_test = pd.read_csv("X_test.csv")

print("--- Predictores:")
print(predictores)
print("\n--- Variable independiente:")
print(var_ind)

# Exploración de los datos: distribución del peso de los bebés al nacer

Estadísticos de la variable a predecir:

In [ ]:
media = np.mean(var_ind)
varianza = np.var(var_ind)
desviacion = np.std(var_ind)
mediana = np.median(var_ind)

print("- Media:", media)
print("- Mediana:", mediana)
print("- Varianza:", varianza)
print("- Desviación estándar:", desviacion)

Verificación de distribución normal por los tests de Kolmogorov-Smirnov y Shapiro-Wilk (los respectivos códigos en Python fueron escritos con asistencia de ChatGPT):

In [ ]:
resultado = stats.kstest(
    var_ind,
    'norm',
    args=(np.mean(var_ind), np.std(var_ind))
)

print("- Estadístico K-S:", resultado.statistic)
print("- p-valor:", resultado.pvalue)

# Shapiro-Wilk
resultado = stats.shapiro(var_ind)

print("\n- Estadístico Shapiro-Wilk:", resultado.statistic)
print("- p-valor:", resultado.pvalue)

En ambos tests, el p-value resulta < 0,05, por lo que se rechaza la hipótesis nula que plantea que los datos siguen una distribución normal.

## Histograma

In [ ]:
plt.hist(var_ind, bins=70, range=[0.5, 14], rwidth=0.85)

plt.xlabel("Peso del bebé al nacer (libras)")
plt.ylabel("Frecuencia")
plt.title("Distribución del peso de neonatos")

plt.show()

Visualmente, el histograma se aproxima a una distribución normal. Es bastante simétrico, algo que también se interpreta al comparar la media con la mediana.

# Análisis de distribuciones de los predictores

## Predictores numéricos: fage, mage, visits y gained

Caracterización:

In [ ]:
print(predictores[["fage", "mage", "visits", "gained"]].describe())

predictores[["fage", "mage", "visits", "gained"]].hist(
    bins=20,
    figsize=(10, 8)
)

plt.show()

Análisis de normalidad (test de Shapiro-Wilk):

In [ ]:
resultado_fage = stats.shapiro(predictores["fage"])
resultado_mage = stats.shapiro(predictores["mage"])
resultado_visits = stats.shapiro(predictores["visits"])
resultado_gained = stats.shapiro(predictores["gained"])

print("- fage:", resultado_fage)
print("- mage:", resultado_mage)
print("- visits:", resultado_visits)
print("- gained:", resultado_gained)

Ninguno de los predictores (numéricos) sigue una distribución normal (p < 0,05). Sin embargo, tanto de los gráficos como de la comparación de las medias y medianas surge que estas variables siguen distribuciones bastante simétricas.

## Predictores categóricos: mature, sex, habit, marital y whitemom

In [ ]:
variables_cat = ["mature", "sex", "habit", "marital", "whitemom"]

for variable in variables_cat:
    #print(variable)
    print(predictores[variable].value_counts())
    print()

# Análisis de la distribución de la variable independiente según cada predictor categórico

Se pidió a ChatGPT que escribiera el código en Python para la clasificación y obtención de parámetros y análisis estadístico (t-test). Importante: para los tests estadísticos se asumió homocedasticidad en todos los casos.

## Influencia del sexo del bebé en su peso al nacer

Se ven diferencias en las medias y medianas entre los distintos sexos:

In [ ]:
plt.boxplot(
    [
        var_ind[predictores["sex"] == "female"],
        var_ind[predictores["sex"] == "male"]
    ],
    labels=["Female", "Male"]
)

plt.xlabel("Sexo")
plt.ylabel("Peso del bebé al nacer (libras)")
plt.title("Distribución del peso al nacer según sexo")

plt.show()

print("Peso según sexo:")

print("\nFemale:")
print(var_ind[predictores["sex"] == "female"].describe())

print("\nMale:")
print(var_ind[predictores["sex"] == "male"].describe())

resultado_ttest = stats.ttest_ind(
    var_ind[predictores["sex"] == "female"],
    var_ind[predictores["sex"] == "male"]
)

print("- Estadístico t:", resultado_ttest.statistic)
print("- p-value:", resultado_ttest.pvalue)

Efectivamente, el peso de los varones es significativamente mayor al de las mujeres en esta muestra al momento de nacer (p < 0,001).

## Influencia de la edad de la madre en el peso del bebé al nacer

In [ ]:
plt.boxplot(
    [
        var_ind[predictores["mature"] == "younger mom"],
        var_ind[predictores["mature"] == "mature mom"]
    ],
    labels=["Younger mom", "Mature mom"]
)

plt.xlabel("Madurez materna")
plt.ylabel("Peso del bebé al nacer (libras)")
plt.title("Distribución del peso al nacer según madurez materna")

plt.show()

print("Peso según madurez materna:")

print("\nYounger mom:")
print(var_ind[predictores["mature"] == "younger mom"].describe())

print("\nMature mom:")
print(var_ind[predictores["mature"] == "mature mom"].describe())

resultado_ttest = stats.ttest_ind(
    var_ind[predictores["mature"] == "younger mom"],
    var_ind[predictores["mature"] == "mature mom"]
)

print("- Estadístico t:", resultado_ttest.statistic)
print("- p-value:", resultado_ttest.pvalue)

Se encontraron diferencias estadísticamente significativas en el peso al nacer según la edad materna (p < 0,05). Los hijos de madres de 35 años o más presentaron un mayor peso promedio que los hijos de madres menores de 35 años.

## Distribución del peso según hábito de fumar

In [ ]:
plt.boxplot(
    [
        var_ind[predictores["habit"] == "nonsmoker"],
        var_ind[predictores["habit"] == "smoker"]
    ],
    labels=["Nonsmoker", "Smoker"]
)

plt.xlabel("Hábito de fumar")
plt.ylabel("Peso del bebé al nacer (libras)")
plt.title("Distribución del peso al nacer según hábito de fumar")

plt.show()

print("Peso según hábito de fumar:")

print("\nNonsmoker:")
print(var_ind[predictores["habit"] == "nonsmoker"].describe())

print("\nSmoker:")
print(var_ind[predictores["habit"] == "smoker"].describe())

resultado_ttest = stats.ttest_ind(
    var_ind[predictores["habit"] == "nonsmoker"],
    var_ind[predictores["habit"] == "smoker"]
)

print("- Estadístico t:", resultado_ttest.statistic)
print("- p-value:", resultado_ttest.pvalue)

Nuevamente, el hecho de que la madre fume afecta el peso del bebé al nacer (p < 0,001). En este sentido, los bebés de madres fumadoras presentaron un peso promedio significativamente menor respecto de aquellos de madres no fumadoras.

## Influencia del estado civil de los progenitores en el peso del neonato

In [ ]:
plt.boxplot(
    [
        var_ind[predictores["marital"] == "married"],
        var_ind[predictores["marital"] == "not married"]
    ],
    tick_labels=["Married", "Not married"]
)

plt.xlabel("Estado civil")
plt.ylabel("Peso del bebé al nacer (libras)")
plt.title("Distribución del peso al nacer según estado civil")

plt.show()

print("Peso según estado civil:")

print("\nMarried:")
print(var_ind[predictores["marital"] == "married"].describe())

print("\nNot married:")
print(var_ind[predictores["marital"] == "not married"].describe())

resultado_ttest = stats.ttest_ind(
    var_ind[predictores["marital"] == "married"],
    var_ind[predictores["marital"] == "not married"]
)

print("- Estadístico t:", resultado_ttest.statistic)
print("- p-value:", resultado_ttest.pvalue)

El estado marital de la madre no afectaría el peso del bebé al nacer.

## Distribución del peso según "raza" de la madre

In [ ]:
plt.boxplot(
    [
        var_ind[predictores["whitemom"] == "white"],
        var_ind[predictores["whitemom"] == "not white"]
    ],
    tick_labels=["White", "Not white"]
)

plt.xlabel("Raza materna")
plt.ylabel("Peso del bebé al nacer (libras)")
plt.title("Distribución del peso al nacer según raza materna")

plt.show()

print("Peso según raza materna:")

print("\nWhite:")
print(var_ind[predictores["whitemom"] == "white"].describe())

print("\nNot white:")
print(var_ind[predictores["whitemom"] == "not white"].describe())

resultado_ttest = stats.ttest_ind(
    var_ind[predictores["whitemom"] == "white"],
    var_ind[predictores["whitemom"] == "not white"]
)

print("- Estadístico t:", resultado_ttest.statistic)
print("- p-value:", resultado_ttest.pvalue)

Se encontraron diferencias estadísticamente significativas en el peso al nacer según la categoría de raza materna. Los hijos de madres clasificadas como white presentaron un peso promedio mayor que aquellos de madres clasificadas como not white (p < 0,001).

### En resumen

Los predictores categóricos sex, mature, habit y whitemom influyen significativamente en el peso del bebé (p < 0,05). No hay evidencia suficiente para afirmar que el estado marital también lo haga.

# Análisis de correlación entre los predictores (individualmente) y el peso al nacer

## Edad del padre y peso del hijo al nacer

In [ ]:
plt.scatter(predictores["fage"], var_ind)

plt.xlabel("Edad del padre (años)")
plt.ylabel("Peso del bebé al nacer (libras)")
plt.title("Relación entre edad paterna y peso al nacer")

print("\nCorrelación de Pearson:")

plt.show()

resultado = stats.pearsonr(predictores["fage"], var_ind)

print("- Coeficiente de correlación:", resultado.statistic)
print("- p-value:", resultado.pvalue)

La correlación fue positiva, pero de muy baja magnitud. El p-value indica que no hay correlación lineal estadísticamente significativa.

## Edad de la madre y peso del hijo al nacer

In [ ]:
plt.scatter(predictores["mage"], var_ind)

plt.xlabel("Edad de la madre (años)")
plt.ylabel("Peso del bebé al nacer (libras)")
plt.title("Relación entre edad materna y peso al nacer")

plt.show()

print("\nCorrelación de Pearson:")

resultado = stats.pearsonr(predictores["mage"], var_ind)

print("- Coeficiente de correlación:", resultado.statistic)
print("- p-value:", resultado.pvalue)

Existe una correlación positiva y estadísticamente significativa entre la edad materna y el peso promedio del bebé al nacer (p < 0,01). Sin embargo, la magnitud de la correlación fue baja, lo que indica una asociación lineal débil.

## Relación entre el número de visitas al obstetra y el peso del hijo al nacer

In [ ]:
plt.scatter(predictores["visits"], var_ind)

plt.xlabel("Cantidad de visitas al obstetra")
plt.ylabel("Peso del bebé al nacer (libras)")
plt.title("Relación entre la cantidad de visitas al obstetra y peso al nacer")

plt.show()

print("\nCorrelación de Pearson:")

resultado = stats.pearsonr(predictores["visits"], var_ind)

print("- Coeficiente de correlación:", resultado.statistic)
print("- p-value:", resultado.pvalue)

No se observó una correlación lineal estadísticamente significativa entre la cantidad de visitas al obstetra y el peso al nacer. La correlación es positiva, pero de muy baja magnitud.

## Correlación entre el peso corporal ganado por la madre y el peso del hijo al nacer

In [ ]:
plt.scatter(predictores["gained"], var_ind)

plt.xlabel("Incremento de peso corporal durante el embarazo")
plt.ylabel("Peso del bebé al nacer (libras)")
plt.title("Relación entre el peso corporal ganado por la madre durante el embarazo y peso al nacer")

plt.show()

print("\nCorrelación de Pearson:")

resultado = stats.pearsonr(predictores["gained"], var_ind)

print("- Coeficiente de correlación:", resultado.statistic)
print("- p-value:", resultado.pvalue)

Se observó una correlación positiva y estadísticamente significativa entre el aumento de peso materno y el peso al nacer (p < 0,01). No obstante, la magnitud de la correlación fue muy baja, indicando una asociación lineal débil entre ambas variables.

# Construcción del modelo

Primeramente, las variables categóricas se convierten en binarias y se agregan como nuevas series al DataFrame de predictores.

In [ ]:
predictores["mature_bin"] = predictores["mature"].map({
    "younger mom": 0,
    "mature mom": 1
})

predictores["sex_bin"] = predictores["sex"].map({
    "female": 0,
    "male": 1
})

predictores["habit_bin"] = predictores["habit"].map({
    "nonsmoker": 0,
    "smoker": 1
})

predictores["marital_bin"] = predictores["marital"].map({
    "not married": 0,
    "married": 1
})

predictores["whitemom_bin"] = predictores["whitemom"].map({
    "not white": 0,
    "white": 1
})

Luego, se dividen (aleatoriamente) los datos de modo de usar una parte (mayoritaria) para el entrenamiento -planteo del modelo-, y los restantes para la validación de la regresión lograda.

In [ ]:
from sklearn.model_selection import train_test_split

X_entrenamiento, X_validación, y_entrenamiento, y_validación = train_test_split(
    predictores,
    var_ind,
    test_size=0.20,
    random_state=42
)

print("Tamaño de X_entrenamiento:", X_entrenamiento.shape)
print("Tamaño de X_validación:", X_validación.shape)

print("\nTamaño de y_entrenamiento:", y_entrenamiento.shape)
print("Tamaño de y_validación:", y_validación.shape)

# Modelos

## Modelo 1

Modelo lineal con predictores: fage, mage, visits, gained, sex_bin, habit_bin, marital_bin, whitemom_bin, mature_bin

In [ ]:
X = np.array(X_entrenamiento[["fage", "mage", "visits", "gained","sex_bin", "habit_bin", "marital_bin", "whitemom_bin", "mature_bin"]])
y = np.array(y_entrenamiento)
reg = LinearRegression()
reg.fit(X,y)
y_pred = reg.predict(X)
MAE1 = mean_absolute_error(y, y_pred)
R21 = r2_score(y, y_pred)

print("MAE:", MAE1)
print("R²:", R21)

## Modelo 2

Modelo lineal con predictores: mage, visits, gained, sex_bin, habit_bin, whitemom_bin

In [ ]:
X = np.array(X_entrenamiento[["mage", "visits", "gained","sex_bin", "habit_bin", "whitemom_bin"]])
y = np.array(y_entrenamiento)
reg = LinearRegression()
reg.fit(X,y)
y_pred = reg.predict(X)
MAE2 = mean_absolute_error(y, y_pred)
R22 = r2_score(y, y_pred)

print("MAE:", MAE2)
print("R²:", R22)

## Modelo 3

Modelo lineal con predictores: mage, gained, habit_bin, whitemom_bin

In [ ]:
X = np.array(X_entrenamiento[["mage","gained","habit_bin", "whitemom_bin"]])
y = np.array(y_entrenamiento)
reg = LinearRegression()
reg.fit(X,y)
y_pred = reg.predict(X)
MAE3 = mean_absolute_error(y, y_pred)
R23 = r2_score(y, y_pred)

print("MAE:", MAE3)
print("R²:", R23)

## Modelo 4

Modelo lineal con predictores: mage, gained, whitemom_bin

In [ ]:
X = np.array(X_entrenamiento[["mage","gained", "whitemom_bin"]])
y = np.array(y_entrenamiento)
reg = LinearRegression()
reg.fit(X,y)
y_pred = reg.predict(X)
MAE4 = mean_absolute_error(y, y_pred)
R24 = r2_score(y, y_pred)

print("MAE:", MAE4)
print("R²:", R24)

## Modelo 5

Modelo lineal con predictores: mage, gained, whitemom_bin

In [ ]:
visitas_seguras = X_entrenamiento["visits"].replace(0.0, 1.0)

X_entrenamiento["visits/mage"] = X_entrenamiento["visits"] / X_entrenamiento["mage"]
X_entrenamiento["gained/mage"] = X_entrenamiento["gained"] / X_entrenamiento["mage"]
X_entrenamiento["gained/visits"] = X_entrenamiento["gained"] / visitas_seguras
X_entrenamiento["habit_bin*gained"] = X_entrenamiento["habit_bin"] * X_entrenamiento["gained"]
X_entrenamiento["sex_bin*gained"] = X_entrenamiento["sex_bin"] * X_entrenamiento["gained"]
X_entrenamiento["visits*mage"] = X_entrenamiento["visits"] * X_entrenamiento["mage"]

X = np.array(X_entrenamiento[["visits/mage", "gained", "gained/visits", "sex_bin*gained", "habit_bin*gained", "visits*mage"]])
y = np.array(y_entrenamiento)
reg = LinearRegression()
reg.fit(X,y)
y_pred = reg.predict(X)
MAE5 = mean_absolute_error(y, y_pred)
R25 = r2_score(y, y_pred)

print("MAE:", MAE5)
print("R²:", R25)